In [11]:
import sys, os

PROJECT_ROOT = os.path.abspath("..")  # because notebook is inside /notebooks
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)
print("Current working dir:", os.getcwd())


Project root added: f:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids
Current working dir: f:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids\notebooks


In [12]:
import pandas as pd
import numpy as np

from src.preprocess import preprocess_data
from src.train_isolation_forest import train_isolation_forest, save_model
from src.evaluate import get_anomaly_scores, choose_threshold, evaluate_model

print("All imports OK ✅")


All imports OK ✅


In [18]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]


In [19]:
import pandas as pd
import os

train_path = os.path.join(PROJECT_ROOT, "data", "raw", "KDDTrain+.txt")
test_path  = os.path.join(PROJECT_ROOT, "data", "raw", "KDDTest+.txt")

df_train = pd.read_csv(train_path, names=columns)
df_test  = pd.read_csv(test_path, names=columns)

print(df_train.shape, df_test.shape)


(125973, 43) (22544, 43)


In [20]:
def decode_bytes_df(df):
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
    return df

df_train = decode_bytes_df(df_train)
df_test = decode_bytes_df(df_test)

print("Decoded byte columns ✅")


Decoded byte columns ✅


In [21]:
df_train["is_attack"] = df_train["label"].apply(lambda x: 0 if x == "normal" else 1)
df_test["is_attack"] = df_test["label"].apply(lambda x: 0 if x == "normal" else 1)

print("Train is_attack:\n", df_train["is_attack"].value_counts())
print("Test is_attack:\n", df_test["is_attack"].value_counts())


Train is_attack:
 is_attack
0    67343
1    58630
Name: count, dtype: int64
Test is_attack:
 is_attack
1    12833
0     9711
Name: count, dtype: int64


In [22]:
X_train_processed, X_test_processed, preprocessor = preprocess_data(df_train, df_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape :", X_test_processed.shape)


X_train_processed shape: (125973, 77)
X_test_processed shape : (22544, 77)


In [23]:
X_train_normal = X_train_processed[df_train["is_attack"] == 0]
print("Normal samples used for training:", X_train_normal.shape)

model = train_isolation_forest(X_train_normal)
save_model(model)

print("Model trained + saved ✅")


Normal samples used for training: (67343, 77)
Model trained + saved ✅


In [24]:
train_scores = get_anomaly_scores(model, X_train_processed)
threshold = choose_threshold(train_scores, percentile=95)

print("Threshold (95th percentile):", threshold)


Threshold (95th percentile): 0.14115964232150902


In [25]:
scores, preds, cm, report = evaluate_model(
    model,
    X_test_processed,
    df_test["is_attack"],
    threshold
)

print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)


Confusion Matrix:
 [[ 9702     9]
 [11904   929]]

Classification Report:
               precision    recall  f1-score   support

           0     0.4490    0.9991    0.6196      9711
           1     0.9904    0.0724    0.1349     12833

    accuracy                         0.4716     22544
   macro avg     0.7197    0.5357    0.3773     22544
weighted avg     0.7572    0.4716    0.3437     22544



In [26]:
tn, fp, fn, tp = cm.ravel()

print("TP (attacks caught):", tp)
print("FN (missed attacks):", fn)
print("FP (false alerts):", fp)
print("TN (normal traffic):", tn)


TP (attacks caught): 929
FN (missed attacks): 11904
FP (false alerts): 9
TN (normal traffic): 9702


In [27]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files",
    "num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate",
    "srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]


In [28]:
df_train = pd.read_csv("F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\\KDDTrain+.txt", names= columns)
df_test = pd.read_csv("F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\KDDTest+.txt", names= columns)

<>:2: SyntaxWarning: invalid escape sequence '\K'
<>:2: SyntaxWarning: invalid escape sequence '\K'
C:\Users\Advait\AppData\Local\Temp\ipykernel_33220\39303604.py:2: SyntaxWarning: invalid escape sequence '\K'
  df_test = pd.read_csv("F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\KDDTest+.txt", names= columns)


In [29]:
print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)

df_train.head()


Train shape: (125973, 43)
Test shape : (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [30]:
df_train['label'].value_counts().head(10)


label
normal         67343
neptune        41214
satan           3633
ipsweep         3599
portsweep       2931
smurf           2646
nmap            1493
back             956
teardrop         892
warezclient      890
Name: count, dtype: int64

In [31]:
df_train['is_attack'] = df_train['label'].apply(lambda x:0 if x=='normal' else 1)
df_test['is_attack'] = df_test['label'].apply(lambda x:0 if x=='normal' else 1)

df_train['is_attack'].value_counts()
df_test['is_attack'].value_counts()


is_attack
1    12833
0     9711
Name: count, dtype: int64

In [32]:
df_train = df_train.drop(columns='difficulty')
df_test = df_test.drop(columns='difficulty')

In [33]:
X_train_processed, X_test_processed, preprocessor = preprocess_data(
    df_train,
    df_test
)

print(X_train_processed.shape)
print(X_test_processed.shape)


(125973, 77)
(22544, 77)


In [34]:
from src.train_isolation_forest import train_isolation_forest, save_model

X_train_normal = X_train_processed[df_train["is_attack"] == 0]

model = train_isolation_forest(X_train_normal)
save_model(model)

print("Saved ✅")





Saved ✅


In [35]:
from src.train_isolation_forest import train_isolation_forest, save_model

X_train_normal = X_train_processed[df_train['is_attack'] == 0]

model = train_isolation_forest(X_train_normal)
save_model(model)

print("Model trained + saved ✅")


Model trained + saved ✅


In [ ]:
from src.evaluate import get_anomaly_scores, choose_threshold

train_scores = get_anomaly_scores(model, X_train_normal)
threshold = choose_threshold(train_scores, percentile=95)


print("Threshold:", threshold)


Threshold: 0.14115964232150902


In [37]:
from src.evaluate import evaluate_model

scores, preds, cm, report = evaluate_model(
    model,
    X_test_processed,
    df_test['is_attack'],
    threshold
)

print("Confusion Matrix:\n", cm)
print("\nReport:\n", report)


Confusion Matrix:
 [[ 9702     9]
 [11904   929]]

Report:
               precision    recall  f1-score   support

           0     0.4490    0.9991    0.6196      9711
           1     0.9904    0.0724    0.1349     12833

    accuracy                         0.4716     22544
   macro avg     0.7197    0.5357    0.3773     22544
weighted avg     0.7572    0.4716    0.3437     22544



In [38]:
for p in [80, 85, 90, 92, 95, 97, 99]:
    t = choose_threshold(train_scores, percentile=p)
    _, _, cm_temp, rep_temp = evaluate_model(model, X_test_processed, df_test["is_attack"], t)

    tn, fp, fn, tp = cm_temp.ravel()
    recall = tp / (tp + fn)
    fpr = fp / (fp + tn)

    print(f"\nPercentile = {p}")
    print("CM:", cm_temp)
    print(f"Attack Recall: {recall:.4f}")
    print(f"False Positive Rate: {fpr:.4f}")



Percentile = 80
CM: [[ 9695    16]
 [10702  2131]]
Attack Recall: 0.1661
False Positive Rate: 0.0016

Percentile = 85
CM: [[ 9698    13]
 [10973  1860]]
Attack Recall: 0.1449
False Positive Rate: 0.0013

Percentile = 90
CM: [[ 9701    10]
 [11340  1493]]
Attack Recall: 0.1163
False Positive Rate: 0.0010

Percentile = 92
CM: [[ 9701    10]
 [11584  1249]]
Attack Recall: 0.0973
False Positive Rate: 0.0010

Percentile = 95
CM: [[ 9702     9]
 [11904   929]]
Attack Recall: 0.0724
False Positive Rate: 0.0009

Percentile = 97
CM: [[ 9705     6]
 [12084   749]]
Attack Recall: 0.0584
False Positive Rate: 0.0006

Percentile = 99
CM: [[ 9710     1]
 [12502   331]]
Attack Recall: 0.0258
False Positive Rate: 0.0001
